# Deploy `mistralai/Devstral-Small-2-24B-Instruct-2512` on SageMaker with the vLLM DLC

[Devstral-Small-2](https://huggingface.co/mistralai/Devstral-Small-2-24B-Instruct-2512) is a 24B agentic coding model (FP8, 256K context) from Mistral AI. This notebook deploys it to a SageMaker real-time endpoint using the [vLLM Deep Learning Container](https://aws.github.io/deep-learning-containers/reference/available_images/#vllm) and the Inference Component (IC) pattern, then runs simple chat, tool-calling, and streaming tests against it.

## Prerequisites

- SageMaker + GPU quota in the target region
- IAM role with SageMaker + ECR + CloudWatch Logs permissions (auto-detected in Studio)
- *(Optional)* Hugging Face token — the [model repo](https://huggingface.co/mistralai/Devstral-Small-2-24B-Instruct-2512) is **public**, but setting `HF_TOKEN` avoids anonymous rate limits and speeds up the weight download from the Hub

In [ ]:
%pip install --upgrade boto3 huggingface_hub

In [ ]:
import time
import re
import json
import os
import boto3
from IPython.display import display, Markdown, clear_output

region = "us-east-2"

boto_session = boto3.Session(region_name=region)
sm = boto_session.client("sagemaker")
sm_runtime = boto_session.client("sagemaker-runtime")

print(f"Region: {region}")

In [ ]:
def get_sagemaker_role():
    """Return the IAM role ARN that this notebook is running under.

    Converts an STS assumed-role ARN to the underlying IAM role ARN, looking up
    the role's actual path (e.g. /service-role/) — Studio-created execution roles
    live at /service-role/ and SageMaker rejects an ARN with the wrong path.
    """
    sts = boto3.client('sts')
    caller = sts.get_caller_identity()
    m = re.match(r"^arn:aws:sts::\d+:assumed-role/([^/]+)/", caller['Arn'])
    if not m:
        return caller['Arn']
    return boto3.client('iam').get_role(RoleName=m.group(1))['Role']['Arn']


def wait_for_endpoint(endpoint_name, sleep_time=30):
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    dots = ""
    while status in ("Creating", "Updating"):
        time.sleep(sleep_time)
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        dots += "."
        clear_output(wait=True)
        print(f"Waiting for endpoint '{endpoint_name}' {dots}")
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")
    if status != "InService":
        reason = sm.describe_endpoint(EndpointName=endpoint_name).get("FailureReason", "")
        raise RuntimeError(f"Endpoint failed: {status}\n{reason}")


def wait_for_ic(ic_name, sleep_time=20):
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    dots = ""
    while status in ("Creating", "Updating"):
        time.sleep(sleep_time)
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        dots += "."
        clear_output(wait=True)
        print(f"Waiting for IC '{ic_name}' {dots}")
    print(f"IC: '{ic_name}', Status: '{status}'")
    if status != "InService":
        reason = sm.describe_inference_component(InferenceComponentName=ic_name).get("FailureReason", "")
        raise RuntimeError(f"IC failed: {status}\n{reason}")

In [ ]:
# Overwrite with your role ARN if wanted
role = None
if role is None:
    role = get_sagemaker_role()
print(f"Role: {role}")

## Hugging Face token (optional)

The model repo is public, so a token isn't required. Setting one avoids anonymous rate limits and gives faster downloads — handy when the container pulls ~24 GB of weights on cold start.

In [ ]:
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""
print("HF_TOKEN set" if hf_token else "No HF_TOKEN — downloading anonymously (slower, subject to rate limits)")

## Configuration

See the [vLLM DLC available images](https://aws.github.io/deep-learning-containers/reference/available_images/#vllm) list for published tags.

**Instance options** (both have 4 GPUs, matching the TP size of 4 below):

- `ml.g5.12xlarge` — 4× A10G (24 GB each, 96 GB total). A10G is SM 8.6 and does **not** support native FP8, so the container runs the FP8 weights via emulation (weight-only FP8 → BF16/FP16 compute). Works fine for this 24B model, just slower than L40S. Cheaper.
- `ml.g6e.12xlarge` — 4× L40S (48 GB each, 192 GB total, SM 8.9). Native FP8 (E4M3) compute — highest throughput. Pricier but recommended for latency-sensitive use.

Uncomment the instance line you want below.

**Env vars**: the `SM_VLLM_*` prefix is handled by the container's entrypoint script, which strips the prefix, lowercases, and passes each as a flag to `vllm serve`. E.g. `SM_VLLM_TOOL_CALL_PARSER=mistral` becomes `--tool-call-parser mistral`.

In [ ]:
# See docs for latest vLLM images: https://aws.github.io/deep-learning-containers/reference/available_images/#vllm
VLLM_DLC_TAG = "0.19.1-gpu-py312-cu129-ubuntu22.04-sagemaker"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:{VLLM_DLC_TAG}"

# Pick one:
instance = {"type": "ml.g5.12xlarge",  "num_gpu": 4}  # 4x A10G — FP8 via emulation, cheaper
# instance = {"type": "ml.g6e.12xlarge", "num_gpu": 4}  # 4x L40S — native FP8, faster

model_id = "mistralai/Devstral-Small-2-24B-Instruct-2512"

deploy_tag = time.strftime("%y%m%d-%H%M%S")
model_name = f"devstral-small-2-{deploy_tag}"
endpoint_name = f"devstral-small-2-{deploy_tag}"
endpoint_config_name = f"devstral-small-2-{deploy_tag}"
ic_name = f"devstral-small-2-ic-{deploy_tag}"
variant_name = "main"
timeout = 1800

# The vLLM DLC entrypoint translates SM_VLLM_<FOO_BAR> → --foo-bar for `vllm serve`.
# HF_TOKEN (no prefix) is read directly by huggingface_hub inside the container.
env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_TRUST_REMOTE_CODE": "true",
    # Mistral-specific loader flags — required for Devstral / Magistral / Voxtral.
    "SM_VLLM_TOKENIZER_MODE": "mistral",
    "SM_VLLM_CONFIG_FORMAT": "mistral",
    "SM_VLLM_LOAD_FORMAT": "mistral",
    # Tool calling (OpenAI-compatible on the wire).
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "mistral",
}
if hf_token:
    env["HF_TOKEN"] = hf_token

print(f"Image:    {inference_image}")
print(f"Instance: {instance['type']} (num_gpu={instance['num_gpu']})")
print(f"Model:    {model_id}")
print(f"Endpoint: {endpoint_name}")

## Deploy: Model + Endpoint + Inference Component

The endpoint is created with an empty production variant (no `ModelName`) — the IC attaches the model after the instance is warm. `NumberOfAcceleratorDevicesRequired` reserves whole GPUs for the IC; `MinMemoryRequiredInMb` is a host-RAM floor, not GPU memory.

In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"Image": inference_image, "Environment": env},
)
print(f"✓ Model:    {model_name}")

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            # InferenceAmiVersion necessary to run on p4 and g5 instances.
            # See docs: https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_ProductionVariant.html
            "InferenceAmiVersion": "al2-ami-sagemaker-inference-gpu-3-1", 
            "ManagedInstanceScaling": {
                "Status": "ENABLED",
                "MinInstanceCount": 1,
                "MaxInstanceCount": 1,
            },
        },
    ],
)

sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
print(f"Creating endpoint '{endpoint_name}' (~5-10 min to pull image and boot instance)...")
wait_for_endpoint(endpoint_name)

In [ ]:
sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": timeout,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 1024,
            "NumberOfAcceleratorDevicesRequired": instance["num_gpu"],
        },
    },
    RuntimeConfig={"CopyCount": 1},
)
print(f"✓ IC:       {ic_name} (waiting for container to start, ~3-8 min)")
wait_for_ic(ic_name)

## Chat

Devstral's system prompt ships alongside the model as `CHAT_SYSTEM_PROMPT.txt`. Best agentic results come from loading it; recommended sampling `temperature=0.15`.

In [ ]:
from huggingface_hub import hf_hub_download

sys_prompt_path = hf_hub_download(
    repo_id=model_id,
    filename="CHAT_SYSTEM_PROMPT.txt",
    token=hf_token or None,
)
with open(sys_prompt_path) as f:
    SYSTEM_PROMPT = f.read()
print(f"System prompt loaded: {len(SYSTEM_PROMPT)} chars")

In [ ]:
payload = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Write a Python function that checks whether a string is a palindrome, ignoring case and non-alphanumerics."},
    ],
    "temperature": 0.15,
    "max_tokens": 512,
}

start = time.time()
res = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(payload),
    ContentType="application/json",
)
response = json.loads(res["Body"].read().decode("utf8"))
print(f"✅ Response time: {time.time()-start:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))
print(f"\n---\nusage: {response['usage']}")

## Tool calling

With `SM_VLLM_TOOL_CALL_PARSER=mistral` + `SM_VLLM_ENABLE_AUTO_TOOL_CHOICE=true`, Devstral emits OpenAI-compatible `tool_calls`.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "git_clone",
            "description": "Clone a git repository to the local workspace",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "HTTPS URL of the repo to clone"},
                },
                "required": ["url"],
            },
        },
    }
]

payload = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Clone https://github.com/huggingface/transformers so we can explore the codebase."},
    ],
    "tools": tools,
    "tool_choice": "auto",
    "temperature": 0.15,
}

res = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(payload),
    ContentType="application/json",
)
response = json.loads(res["Body"].read().decode("utf8"))
print(json.dumps(response["choices"][0]["message"], indent=2))

## Streaming

In [ ]:
stream_payload = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Explain how a B-tree index speeds up range queries, in 4 short bullets."},
    ],
    "temperature": 0.15,
    "max_tokens": 256,
    "stream": True,
}

res = sm_runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(stream_payload),
    ContentType="application/json",
)

buffer = ""
for event in res["Body"]:
    if "PayloadPart" not in event:
        continue
    buffer += event["PayloadPart"]["Bytes"].decode("utf-8")
    lines = buffer.split("\n")
    buffer = lines[-1]
    for line in lines[:-1]:
        line = line.strip()
        if not line or line == "data: [DONE]":
            continue
        if line.startswith("data: "):
            line = line[6:]
        try:
            chunk = json.loads(line)
            delta = chunk["choices"][0].get("delta", {})
            text = delta.get("content") or ""
            if text:
                print(text, end="", flush=True)
        except json.JSONDecodeError:
            pass
print()

## Cleanup

In [ ]:
sm.delete_inference_component(InferenceComponentName=ic_name)
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_model(ModelName=model_name)